## Residency analysis (R vs NON‑R) and CGPA impact

Requirement: compare **Resident (R)** vs **Non‑Resident (NON‑R)** students and quantify impact on CGPA.

This notebook:
- Searches the available datasets for a residency field (or explicit `R`/`NON-R` values)
- If found, performs:
  - Unadjusted CGPA comparison (distributions + mean/median)
  - Adjusted comparison controlling for past performance, attendance, payments, and sponsorship (regression)
- If not found, documents the missing-data limitation and the modeling assumption needed to include residency.

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression

BASE_DIR = Path.cwd()
OUT_DIR = BASE_DIR / "outputs"

model_df = pd.read_csv(OUT_DIR / "modeling_table_student_semester.csv")

model_df.shape

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\final-year-project-data-analysis\\Other_Analysis\\outputs\\modeling_table_student_semester.csv'

In [ ]:
# 1) Look for likely residency columns
res_cols = [c for c in model_df.columns if any(k in c.upper() for k in ["RES", "RESIDENT"]) ]
res_cols

In [ ]:
# 2) Scan object columns for explicit values R / NON-R
obj_cols = [c for c in model_df.columns if model_df[c].dtype == "object"]

hits = []
for c in obj_cols:
    s = model_df[c].dropna().astype(str).str.upper().str.strip()
    # sample unique values cheaply
    uniq = pd.Index(s.unique())
    if ("R" in uniq) or ("NON-R" in uniq) or ("NONR" in uniq) or ("NON RESIDENT" in uniq) or ("RESIDENT" in uniq):
        hits.append((c, sorted(list(uniq.intersection(["R", "NON-R", "NONR", "RESIDENT", "NON RESIDENT"])))) )

hits[:20], len(hits)

## If residency is missing

If `res_cols` and `hits` are empty, the available CSVs do not contain an explicit residency flag.

**Modeling implication:** we cannot estimate the causal/associational effect of residency on CGPA from these files alone.

To include residency in the predictive model, one of these must be provided:
- a `RESIDENCY` column (values like `R`/`NON-R`) in the student master table, or
- a deterministic mapping from an existing identifier to residency (documented business rule), or
- a separate residence/housing dataset keyed by `REG_NO`.

The next cell runs the comparison automatically *if* a residency field is found.

In [ ]:
# Choose residency column if present
residency_col = None

# Prefer explicit columns
for c in res_cols:
    residency_col = c
    break

# Otherwise use first detected hit
if residency_col is None and hits:
    residency_col = hits[0][0]

residency_col

In [ ]:
if residency_col is None:
    print("No explicit residency indicator found in the available modeling table.")
else:
    d = model_df.dropna(subset=["CGPA"]).copy()
    d["RESIDENCY"] = d[residency_col].astype(str).str.upper().str.strip().replace({"NONR": "NON-R", "NON RESIDENT": "NON-R", "RESIDENT": "R"})
    d = d[d["RESIDENCY"].isin(["R", "NON-R"])].copy()

    print("Counts:")
    print(d["RESIDENCY"].value_counts(dropna=False))

    print("\nCGPA summary by residency:")
    print(d.groupby("RESIDENCY")["CGPA"].describe())

    # Adjusted comparison (linear regression with controls)
    feature_cols = [c for c in [
        "prev_cgpa","prev_semester_gpa",
        "present_rate","absent_rate","late_rate",
        "payment_success_rate","total_paid_success",
        "has_sponsorship","sponsor_amount_total",
        "SEMESTER_INDEX","PROGRAM",
    ] if c in d.columns]

    X = d[feature_cols + ["RESIDENCY"]]
    y = d["CGPA"].astype(float)

    num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(d[c])]
    cat_cols = [c for c in feature_cols if c not in num_cols] + ["RESIDENCY"]

    pre = ColumnTransformer([
        ("num", Pipeline([( "imp", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([( "imp", SimpleImputer(strategy="most_frequent")), ("oh", OneHotEncoder(drop=None, handle_unknown="ignore"))]), cat_cols),
    ])

    reg = Pipeline([( "pre", pre), ("lr", LinearRegression())])
    reg.fit(X, y)

    print("\nAdjusted regression fitted.")
    print("Interpretation: the residency effect is encoded in the one-hot coefficient for RESIDENCY (requires extracting feature names).")
